# 🗄️ Delta Lake 3.2: ACID Transactions & Time Travel Deep Dive
### *Demonstrating Write-Ahead Transaction Logs, Versioning & Audit Rollbacks*

This tutorial demonstrates how Delta Lake's `_delta_log` enables reproducible queries, schema enforcement, and point-in-time time-travel rollbacks on streaming mobility data.

In [ ]:
# Conceptual Python walkthrough of Delta Lake Time Travel operations
import pandas as pd
from datetime import datetime, timedelta

# Sample transaction commit log representation
delta_commit_history = [
    {"version": 0, "timestamp": "2026-08-26 10:00:00", "operation": "CREATE TABLE", "rows_affected": 0},
    {"version": 1, "timestamp": "2026-08-26 10:15:00", "operation": "STREAMING WRITE (Bronze)", "rows_affected": 15000},
    {"version": 2, "timestamp": "2026-08-26 10:30:00", "operation": "STREAMING WRITE (Silver)", "rows_affected": 14650},
    {"version": 3, "timestamp": "2026-08-26 11:00:00", "operation": "OPTIMIZE (Z-ORDER)", "rows_affected": 14650},
    {"version": 4, "timestamp": "2026-08-26 11:30:00", "operation": "VACUUM (Retain 168h)", "rows_affected": 0}
]

history_df = pd.DataFrame(delta_commit_history)
print("=== Delta Lake Transaction History (DESCRIBE HISTORY) ===")
print(history_df)

## 1. Time Travel Query Patterns in PySpark SQL

```sql
-- Query Table as of Version 2 (Before 11:00 AM Compaction)
SELECT * FROM fact_traffic VERSION AS OF 2;

-- Query Table as of Specific Timestamp
SELECT * FROM fact_traffic TIMESTAMP AS OF '2026-08-26 10:30:00';

-- Restore Table to Previous Known Good State after Accidental Deletion
RESTORE TABLE fact_traffic TO VERSION AS OF 2;
```